In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

# New simplified imports
from movebench import Study, grouped_nll

# Old vs New API Comparison

### OLD way (verbose):
```python
study = '5a280544-06db-4c85-9713-842547972edd'
df = pd.read_csv(glob.glob(f'mbr_studies_2year_filtered_traintest/{study}*')[0])
df = df[df.set == 'train']
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp'])
ds = ChipDataset(study, idx_to_timestamp=pd.to_datetime(df.timestamp))
```

### NEW way (clean):
```python
study = Study.load('5a280544-06db-4c85-9713-842547972edd')
ds = study.chip_dataset(split='train')
```

---
## List Available Studies

In [ ]:
# List all available studies
available_studies = Study.list_available()
print(f"Found {len(available_studies)} studies")
print(available_studies[:5])

---
## Load a Study and Create Datasets

In [ ]:
# Load a study - all the CSV parsing, timestamp handling, etc. is automatic
study = Study.load('5a280544-06db-4c85-9713-842547972edd')
print(study)

In [ ]:
# Create ChipDataset - just specify the split!
train_ds = study.chip_dataset(split='train')
print(f"Train dataset: {len(train_ds)} samples")

# Load one sample
x, mask, transform, crs = train_ds[0]
print(f"Sample shape: {x.shape}")

---
## Compute Normalization Statistics

### OLD way:
```python
# 30+ lines of boilerplate with manual loops, sum tracking, etc.
```

### NEW way:

In [ ]:
# Compute normalization for all studies that need it
for study_id in Study.list_available():
    try:
        s = Study.load(study_id)
        s.compute_normalization(num_samples=1000)  # Skips if already computed
    except FileNotFoundError:
        print(f"Skipping {study_id} - data files not found")

---
## SSF Dataset

### OLD way:
```python
df = pd.read_csv(glob.glob(f'mbr_studies_2year_filtered_traintest/{study}*')[0])
df = df[df.set == 'train']
sds = SSFDataset(study, df)
```

### NEW way:

In [ ]:
# Create SSF dataset - one line!
ssf_ds = study.ssf_dataset(split='train')
print(f"SSF dataset: {len(ssf_ds)} samples")

# Load one sample
step_pixels, valid_pixels, chip_transform, chip_crs = ssf_ds[0]
print(f"Step pixels shape: {step_pixels.shape}")  # (63, 101) = 63 features, 101 candidates

In [ ]:
# Inspect features for the first candidate
print("Features for first candidate point:")
print(step_pixels[:, 0])
print("\nValid mask:")
print(valid_pixels[:, 0])

---
## Compute and Cache SSF Features

In [ ]:
import os

# Studies with DEM data available
studies_with_data = [
    '5a280544-06db-4c85-9713-842547972edd',
    'fd28aac2-b22f-4368-8638-59906e769ded',
    '28708a9c-5582-490e-9481-c094bdce74c2',
    '338388ea-6483-472c-9cdc-c2997e645df0',
]

for study_id in studies_with_data:
    study = Study.load(study_id)
    
    for split in ['train', 'test']:
        cache_file = f'{study_id}_{split}_xs.pt'
        if os.path.exists(cache_file):
            print(f"Already computed SSF features for {study_id} {split}")
            continue
            
        print(f"Computing SSF features for {study_id} {split}")
        
        # Create dataset with new API
        sds = study.ssf_dataset(split=split)
        
        dataloader = DataLoader(
            sds, 
            batch_size=1,
            shuffle=False,
            num_workers=16,
        )
        
        all_xs = []
        all_masks = []
        all_ys = []
        
        for step_pixels, valid_pixels, _, _ in tqdm(dataloader, total=len(sds)):
            all_xs.append(step_pixels)
            all_masks.append(valid_pixels)
            all_ys.append(torch.tensor([0] * sds.num_candidates + [1]))
        
        # Reshape: (N, C, 101) -> (N*101, C)
        all_xs_t = torch.cat(all_xs).permute(0, 2, 1).reshape(-1, 63)
        all_masks_t = torch.cat(all_masks).permute(0, 2, 1).reshape(-1, 63)
        all_ys_t = torch.cat(all_ys)
        
        torch.save(all_xs_t, f'{study_id}_{split}_xs.pt')
        torch.save(all_masks_t, f'{study_id}_{split}_masks.pt')
        torch.save(all_ys_t, f'{study_id}_{split}_ys.pt')
        
        print(f"Saved {study_id} {split}: {all_xs_t.shape}")

---
## Train SSF Model

In [ ]:
# Load cached features
study_id = 'fd28aac2-b22f-4368-8638-59906e769ded'

X_tr = torch.load(f'{study_id}_train_xs.pt')
masks_tr = torch.load(f'{study_id}_train_masks.pt')
y_tr = torch.load(f'{study_id}_train_ys.pt')

X_te = torch.load(f'{study_id}_test_xs.pt')
masks_te = torch.load(f'{study_id}_test_masks.pt')
y_te = torch.load(f'{study_id}_test_ys.pt')

print(f"Train: {X_tr.shape}, Test: {X_te.shape}")

In [ ]:
# Prepare data for training
def prep_ssf_data(X, masks, y):
    """Prepare SSF data for conditional logistic regression."""
    # Group indices (101 candidates per step)
    starts = torch.arange(0, X.shape[0], 101)
    ends = torch.arange(101, X.shape[0] + 1, 101)
    
    # Fill invalid values with 0 (approximately the mean after standardization)
    X_clean = torch.where(masks, X, torch.tensor(0.0))
    X_clean = torch.nan_to_num(X_clean, nan=0.0, posinf=0.0, neginf=0.0)
    
    return X_clean, y, starts, ends

X_tr, y_tr, starts_tr, ends_tr = prep_ssf_data(X_tr, masks_tr, y_tr)
X_te, y_te, starts_te, ends_te = prep_ssf_data(X_te, masks_te, y_te)

In [ ]:
# Train conditional logistic regression
p = X_tr.shape[1]  # 63 features
beta = torch.zeros(p, dtype=torch.float32, requires_grad=True)
opt = torch.optim.Adam([beta], lr=1e-2, weight_decay=1e-3)

for step in range(1000):
    opt.zero_grad()
    loss = grouped_nll(beta, X_tr, y_tr, starts_tr, ends_tr)
    loss.backward()
    opt.step()
    
    if step % 100 == 0:
        with torch.no_grad():
            nll_tr = grouped_nll(beta, X_tr, y_tr, starts_tr, ends_tr).item()
            nll_te = grouped_nll(beta, X_te, y_te, starts_te, ends_te).item()
            print(f"Step {step:4d} | Train NLL: {nll_tr:.4f} | Test NLL: {nll_te:.4f}")

In [ ]:
# Inspect learned coefficients
from movebench import get_channel_indices

print("Top 10 coefficients by magnitude:\n")
indices = get_channel_indices()
channel_names = []
for name, (start, end) in indices.items():
    for i in range(start, end):
        channel_names.append(f"{name}_{i-start}")

beta_np = beta.detach().numpy()
sorted_idx = abs(beta_np).argsort()[::-1]

for i in sorted_idx[:10]:
    print(f"{channel_names[i]:30s} {beta_np[i]:+.4f}")

---
## Summary: API Improvements

| Task | Old API | New API |
|------|---------|--------|
| List studies | Manual glob | `Study.list_available()` |
| Load trajectories | 4 lines of boilerplate | `study.trajectories` |
| Create ChipDataset | 6 lines | `study.chip_dataset(split='train')` |
| Create SSFDataset | 4 lines | `study.ssf_dataset(split='train')` |
| Compute normalization | 30+ lines | `study.compute_normalization()` |
| Get categorical indices | Magic numbers | `get_categorical_indices()` |